# Noise Pollution Monitor — K-Means Clustering
This notebook builds a starter noise-zone classifier using sound level, location, and time.

**Important:** `data/noise_data.csv` is synthetic demonstration data, not real field measurements. Replace it with calibrated sound-meter readings before drawing real-world conclusions. K-Means discovers clusters; the notebook names them by average sound level.

## Step 1 — Install packages
Run this cell once in your Jupyter environment. If installation finishes, restart the kernel if Jupyter asks you to.

In [ ]:
%pip install pandas numpy scikit-learn joblib matplotlib

## Step 2 — Import libraries and set paths

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import silhouette_score
import joblib

PROJECT_DIR = Path.cwd()
# If your notebook is inside the notebooks folder, use its parent as the project root.
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
DATA_PATH = PROJECT_DIR / "data" / "noise_data.csv"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR)
print("Dataset exists:", DATA_PATH.exists())

## Step 3 — Load the dataset
The required columns are `timestamp`, `location`, and `sound_level_db`. The category is created by K-Means later.

In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()

## Step 4 — Clean data and create time features
We convert timestamp to a datetime, remove invalid rows, and create hour and day-of-week features. Cyclic hour features help represent the fact that 23:00 and 00:00 are close in time.

In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df["sound_level_db"] = pd.to_numeric(df["sound_level_db"], errors="coerce")
df = df.dropna(subset=["timestamp", "location", "sound_level_db"]).copy()
df = df[df["sound_level_db"].between(0, 150)].copy()
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["day_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["day_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)
print("Rows after cleaning:", len(df))
df.head()

## Step 5 — Explore the data

In [ ]:
display(df[["sound_level_db", "location"]].describe(include="all"))
df["sound_level_db"].hist(bins=25)
plt.xlabel("Sound level (dB)")
plt.ylabel("Number of readings")
plt.title("Distribution of sound measurements")
plt.show()

display(df.groupby("location")["sound_level_db"].agg(["count", "mean", "min", "max"]).sort_values("mean"))

## Step 6 — Train K-Means with three clusters
Sound level is the main signal. Location is one-hot encoded, while hour/day are represented with cyclic features. All numerical features are standardized so that one feature does not dominate simply because of its scale.

In [ ]:
numeric_features = ["sound_level_db", "hour_sin", "hour_cos", "day_sin", "day_cos"]
categorical_features = ["location"]

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("location", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("kmeans", KMeans(n_clusters=3, random_state=42, n_init=10))
])

feature_columns = numeric_features + categorical_features
model.fit(df[feature_columns])
df["cluster_id"] = model.predict(df[feature_columns])

cluster_means = df.groupby("cluster_id")["sound_level_db"].mean().sort_values()
cluster_to_category = {
    int(cluster_id): name
    for cluster_id, name in zip(cluster_means.index, ["Quiet Zone", "Moderate Zone", "Noisy Zone"])
}
df["category"] = df["cluster_id"].map(cluster_to_category)

print("Average dB by cluster:")
display(df.groupby(["cluster_id", "category"])["sound_level_db"].agg(["count", "mean", "min", "max"]))

## Step 7 — Review cluster separation
Silhouette score is a clustering diagnostic, not classification accuracy. It may be undefined or misleading when data has very little separation.

In [ ]:
X_transformed = model.named_steps["preprocessor"].transform(df[feature_columns])
score = silhouette_score(X_transformed, df["cluster_id"])
print(f"Silhouette score: {score:.3f} (closer to 1 generally indicates better-separated clusters)")

## Step 8 — Save the model and zone mapping
The Streamlit app will load this artifact. Keep the same dependency versions in your local environment and deployment environment.

In [ ]:
artifact = {
    "model": model,
    "cluster_to_category": cluster_to_category,
    "feature_columns": feature_columns,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "cluster_means_db": {int(k): float(v) for k, v in cluster_means.items()},
}
joblib.dump(artifact, ARTIFACTS_DIR / "noise_kmeans_model.joblib")
df[["timestamp", "location", "sound_level_db", "cluster_id", "category"]].to_csv(
    PROJECT_DIR / "data" / "noise_data_with_categories.csv", index=False
)
print("Saved model:", ARTIFACTS_DIR / "noise_kmeans_model.joblib")
print("Saved categorized data:", PROJECT_DIR / "data" / "noise_data_with_categories.csv")

## Step 9 — Predict a new reading

In [ ]:
def make_time_features(timestamp):
    ts = pd.Timestamp(timestamp)
    return {
        "hour_sin": np.sin(2 * np.pi * ts.hour / 24),
        "hour_cos": np.cos(2 * np.pi * ts.hour / 24),
        "day_sin": np.sin(2 * np.pi * ts.dayofweek / 7),
        "day_cos": np.cos(2 * np.pi * ts.dayofweek / 7),
    }

reading_time = pd.Timestamp.now()
new_reading = {
    "sound_level_db": 67.0,
    "location": "Main Road",
    **make_time_features(reading_time)
}
new_df = pd.DataFrame([new_reading])[feature_columns]
cluster_id = int(model.predict(new_df)[0])
category = cluster_to_category[cluster_id]
print(f"Location: {new_reading['location']}")
print(f"Sound level: {new_reading['sound_level_db']} dB")
print(f"Predicted zone: {category}")

## Step 10 — Next steps
1. Replace the synthetic CSV with actual calibrated readings.
2. Run all notebook cells from top to bottom after replacing the data.
3. Confirm the model file exists in `artifacts/`.
4. Run the Streamlit app from the project root using `streamlit run app.py`.
5. Upload the project to GitHub and deploy through Streamlit Community Cloud.